# Train/Test Split and Overfitting

**DS4DH Practice Pack · Module 07 — Machine Learning and Interpretability**

*Technique:* Holding out data, and why a model's score on its own training data is meaningless

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/07a_train_test_split.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A model evaluated on the data it learned from will always look better than it is.
A sufficiently flexible model can memorise its training set perfectly and have
learned nothing generalisable at all.

The fix is to hide some data from the model and score it on that. This notebook
shows the failure first, so the fix is not just a ritual.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

FEATURES = ['Total', 'renter_owner_gap', 'tot_income', 'log_pop']
RAW_NEEDED = ['Total', 'renter_owner_gap', 'tot_income', 'tot_pop']

feat = base.dropna(subset=RAW_NEEDED).copy()
feat['log_pop'] = np.log10(feat['tot_pop'])

print(f'{len(base)} CSDs -> {len(feat)} with complete data on all four columns')
print(f'{len(base) - len(feat)} dropped (census suppression in small places)')
print()
for city, n in feat['cma'].value_counts().items():
    print(f'  {city:<11} {n:>3} CSDs')

In [ ]:
TARGET = 'Total'
PREDICTORS = ['renter_owner_gap', 'tot_income', 'log_pop']
RANDOM_STATE = 42

X = feat[PREDICTORS]
y = feat[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE)

print(f'training set: {len(X_train)} CSDs')
print(f'test set:     {len(X_test)} CSDs')
print()
print('The test set is not used to fit anything. It exists to be surprised by.')

## Watching a model memorise

A decision tree's `max_depth` controls how flexible it is. Unlimited depth lets
it carve the training data into leaves of one observation each — a perfect score,
and a useless model.

In [ ]:
print(f'{"max_depth":>10}{"train R²":>12}{"test R²":>11}{"gap":>9}')
print('-' * 42)
rows = []
for depth in [1, 2, 3, 4, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    tr = r2_score(y_train, m.predict(X_train))
    te = r2_score(y_test, m.predict(X_test))
    rows.append((depth if depth else 99, tr, te))
    label = str(depth) if depth else 'none'
    print(f'{label:>10}{tr:>12.3f}{te:>11.3f}{tr - te:>9.3f}')
print()
print('Training R² climbs to 1.0. Test R² peaks early and then falls.')
print('The gap between the two columns is overfitting, measured.')

In [ ]:
fig, ax = plt.subplots()
d = [r[0] for r in rows]
ax.plot(d, [r[1] for r in rows], 'o-', label='train R²')
ax.plot(d, [r[2] for r in rows], 'o-', color='#E8663D', label='test R²')
ax.set_xlabel('max_depth (99 = unlimited)')
ax.set_ylabel('R²')
ax.set_title('The training score is not the score')
ax.legend()
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Change `test_size` from `0.25` to `0.4`, then to `0.1`, re-running the table.

With a test set of only 15 CSDs, how much does test R² move? A test score
computed on very few observations is itself very noisy — which is the problem the
next cell addresses.

## One split is not enough

With 155 observations, a single 25% test set is 39 places. Which 39 you happened
to draw matters. Cross-validation splits repeatedly and reports the spread.

In [ ]:
for depth in [2, 3, 5, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
    s = cross_val_score(m, X, y, cv=5, scoring='r2')
    label = str(depth) if depth else 'none'
    print(f'max_depth={label:<5} CV R² = {s.mean():>6.3f} ± {s.std():.3f}'
          f'   folds: {np.round(s, 3)}')
print()
print('The ± is the honest error bar on the score. A model reported without')
print('one is a model reported without evidence that it would hold up.')

In [ ]:
# Instability, shown directly: refit with different splits and watch the score.
scores = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed)
    m = DecisionTreeRegressor(max_depth=3, random_state=RANDOM_STATE).fit(Xtr, ytr)
    scores.append(r2_score(yte, m.predict(Xte)))

scores = np.array(scores)
print(f'20 different random splits, same model and same data:')
print(f'  test R² ranges from {scores.min():.3f} to {scores.max():.3f}')
print(f'  mean {scores.mean():.3f}, sd {scores.std():.3f}')
print()
print('Quoting a single split\'s score is quoting one draw from that range.')

### 🔧 Your turn 2

Add `mean_absolute_error` alongside R² in the depth table.

MAE is in percentage points — the same units as STIR. Which of the two numbers
would you put in front of a housing analyst, and why?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** With `test_size=0.1` the test set is about 15 CSDs and the score
swings wildly between runs — sometimes above the training score, which is
impossible as a real property of the model and purely an artefact of which places
landed in the test set. Small test sets do not just give imprecise estimates; they
give estimates that can point the wrong way.

**Your turn 2.** MAE, because it is interpretable: "the model's prediction of a
municipality's STIR is off by about 2 percentage points on average" is a sentence
a housing analyst can weigh against how much precision they need. R² answers
"what share of variance is explained", which is a modelling question, not a
policy one. Report both; lead with MAE.

</details>

## Where this stops

You now have a defensible way to score a model. The next notebook uses it on a
model actually worth fitting.